In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'https://develop.rdm.example.com/'

idp_name_1 = None
idp_username_institutional_admin = "user_test_admin"
idp_password_institutional_admin = "password_test_admin"
idp_username_2 = "user_test02"
idp_password_2 = "password_test02"
idp_username_3 = "user_test03"
idp_password_3 = "password_test03"
idp_username_4 = "user_test04"
idp_password_4 = "password_test04"
idp_username_5 = "user_test05"
idp_password_5 = "password_test05"

group_a = "GroupA"
group_b = "GroupB"
group_c = "GroupC"
group_d = "GroupD"

today = datetime.now().strftime("%Y%m%d")
rdm_project_name = f"TEST-グループ管理連携機能検証-{today}"
component_title_display_username_2 = f"TEST-グループ管理連携機能検証-{today}-コンポーネント-ユーザー2"
component_title_display_username_4 = f"TEST-グループ管理連携機能検証-{today}-コンポーネント-ユーザー4"
target_storage_name = 'NII Storage'
target_storage_id = 'osfstorage'
delete_project = True
default_result_path = None
close_on_fail = False
transition_timeout = 60000
group_note_text = '※本機能はムーンショット目標2'

In [ ]:
if idp_username_institutional_admin is None:
    idp_username_institutional_admin = input(prompt=f'Username for {idp_username_institutional_admin}')
if idp_password_institutional_admin is None:
    idp_password_institutional_admin = getpass(prompt=f'Password for {idp_username_institutional_admin}@{idp_name_1}')
(len(idp_username_institutional_admin), len(idp_password_institutional_admin))

In [ ]:
if idp_username_2 is None:
    idp_username_2 = input(prompt=f'Username for {idp_name_1}')
if idp_password_2 is None:
    idp_password_2 = getpass(prompt=f'Password for {idp_username_2}@{idp_name_1}')
(len(idp_username_2), len(idp_password_2))

if idp_username_3 is None:
    idp_username_3 = input(prompt=f'Username for {idp_name_1}')
if idp_password_3 is None:
    idp_password_3 = getpass(prompt=f'Password for {idp_username_3}@{idp_name_1}')
(len(idp_username_3), len(idp_password_3))

if idp_username_4 is None:
    idp_username_4 = input(prompt=f'Username for {idp_name_1}')
if idp_password_4 is None:
    idp_password_4 = getpass(prompt=f'Password for {idp_username_4}@{idp_name_1}')
(len(idp_username_4), len(idp_password_4))

if idp_username_5 is None:
    idp_username_5 = input(prompt=f'Username for {idp_name_1}')
if idp_password_5 is None:
    idp_password_5 = getpass(prompt=f'Password for {idp_username_5}@{idp_name_1}')
(len(idp_username_5), len(idp_password_5))

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# グループ権限によるプロジェクトに対する操作

- サブシステム名: グループ管理連携機能
- ページ/アドオン: グループ権限によるプロジェクトへのアクセス
- 機能分類: グループ権限によるプロジェクトに対する操作
- シナリオ名: プロジェクトに対する権限確認
- 用意するテストデータ: アカウント(既存ユーザー1,2,3,4,5)

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## ウェブブラウザの新規プライベートウィンドウで GRDM トップページを表示する

- GRDM トップページが表示されること

In [ ]:
import time

async def _step(page):
    await page.goto(rdm_url)

    # 同意する ボタンが現れるまで待つ
    await expect(page.locator('//button[text() = "同意する"]')).to_be_visible(timeout=transition_timeout)

    # 同意する をクリック
    await page.locator('//button[text() = "同意する"]').click()

    # 同意する が表示されなくなったことを確認
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step)

## RDMIdPを利用し、既存ユーザー2としてログインする

- GRDM ダッシュボードが表示されること
- 「プロジェクトに対するグループ機能①」シートのNo.4で追加したプロジェクトが表示されていること
- 該当プロジェクトの「Groups」項目に既存ユーザー2が所属しているグループ(グループA)が表示されていること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_2, idp_password_2, transition_timeout=transition_timeout)

    # GRDMのボタンが表示されることを確認
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)
    project_locator = page.locator('[data-test-dashboard-item]', has=page.locator('[data-test-dashboard-item-title]', has_text=rdm_project_name))
    await expect(project_locator).to_be_visible(timeout=transition_timeout)
    
    # await expect(project_locator.locator('.di-groups')).to_contain_text(group_a, timeout=transition_timeout)
    await expect(project_locator).to_contain_text(group_a, timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードから「プロジェクトに対するグループ機能①」シートのNo.4で作成したプロジェクトをクリックする

- プロジェクトダッシュボードが表示されること
- プロジェクトダッシュボードの上部メニューに「ファイル」、「Wiki」、「メタデータ」、「エクスポート」、「メンバ」、「グループ」、「アドオン」、「設定」が表示されること(「読込み/書込み」権限)

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]').click()

    await expect(page.locator("#projectSubnav").get_by_role("link", name="ファイル")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="Wiki")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メタデータ")).to_be_visible(timeout=transition_timeout)
    # await expect(page.locator("#projectSubnav").get_by_role("link", name="エクスポート")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メンバー")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True)).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="アドオン")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="設定")).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)

await run_pw(_step)

## 画面上部の「マイプロジェクト」をクリックする

- マイプロジェクトが表示されること
- 「プロジェクトに対するグループ機能①」シートのNo.4で追加したプロジェクトが表示されていること
- 該当プロジェクトの「Groups」項目に既存ユーザー2が所属しているグループ(グループA)が表示されていること

In [ ]:
async def _step(page):
    
    await page.locator('//a[text()="マイプロジェクト"]').click()
    
    #マイプロジェクト一覧が表示されること
    await expect(page.locator("h3", has_text="マイプロジェクト")).to_be_visible(timeout=transition_timeout)

    #プロジェクトが表示されること
    await expect(page.locator(f'//a[@class="fg-file-links" and text()="{rdm_project_name}"]')).to_be_visible(timeout=transition_timeout)

    #グループ(グループA)が表示されていること
    import re

    project_row = page.locator(".tb-row").filter(has=page.locator(f'//a[@class="fg-file-links" and text()="{rdm_project_name}"]'))

    container = project_row.locator(".tb-td.tb-col-3")
    group_span = container.locator("span", has_text=group_a)
    plus_indicator = container.locator("span", has_text=re.compile(r'\+ \d+'))

    # Verify group name existence: either shown directly as a span, or folded into "+N" (overflow)
    # (plus_indicator being visible does not guarantee the target group is included in it,
    #  so check group_span first and only fall back to the +N possibility if not found)
    if await group_span.count() > 0 and await group_span.first.is_visible():
        # Group name span is present in DOM (may be shown in full or ellipsis-truncated)
        await expect(group_span.first).to_be_visible(timeout=transition_timeout)
    elif await plus_indicator.count() > 0 and await plus_indicator.first.is_visible():
        # Overflow groups are collapsed into "+N" (tooltip not checked)
        await expect(plus_indicator.first).to_be_visible(timeout=transition_timeout)
    else:
        # Neither span nor plus_indicator matched, but the text should still exist in the DOM, so verify it
        await expect(container).to_contain_text(group_a, timeout=transition_timeout)

    # Ellipsis (visual truncation) verification always runs as its own case, independent of the existence check above
    is_truncated = await container.evaluate(
        "el => el.scrollWidth > el.clientWidth"
    )

    # Independent signal: check the computed CSS truncation setup itself, not derived from is_truncated
    # (so this assertion can actually fail if the ellipsis CSS is broken/removed by a regression)
    has_ellipsis_style = await container.evaluate(
        "el => { const s = getComputedStyle(el); return s.textOverflow === 'ellipsis' && s.overflow === 'hidden'; }"
    )

    if is_truncated:
        # Text is long and overflows the column -> the ellipsis CSS must actually be applied to visually truncate it
        assert has_ellipsis_style, "Expected ellipsis CSS (text-overflow/overflow) to be applied when text overflows"
    else:
        # Text is short and fits the column -> no truncation needed, full text is shown as-is
        assert not is_truncated, "Expected short text NOT to be truncated (no ellipsis)"
await run_pw(_step)

## マイプロジェクトから「プロジェクトに対するグループ機能①」シートのNo.4で作成したプロジェクトをクリックする

- プロジェクトダッシュボードが表示されること
- プロジェクトダッシュボードの上部メニューに「ファイル」、「Wiki」、「メタデータ」、「エクスポート」、「メンバ」、「グループ」、「アドオン」、「設定」が表示されること(「読込み/書込み」権限)

In [ ]:
async def _step(page):
    
    await page.locator(f'//a[@class="fg-file-links" and text()="{rdm_project_name}"]').click()

    await expect(page.locator("#projectSubnav").get_by_role("link", name="ファイル")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="Wiki")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メタデータ")).to_be_visible(timeout=transition_timeout)
    # await expect(page.locator("#projectSubnav").get_by_role("link", name="エクスポート")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メンバー")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True)).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="アドオン")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="設定")).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)

await run_pw(_step)

## プロジェクトダッシュボードの画面右部の「コンポーネント」の「コンポーネントを追加」をクリックする

- 「新しいコンポーネントを作成する」ダイアログが表示されること

In [ ]:
async def _step(page):
    
    await page.locator('div[data-target="#addSubComponent"]').click()

    await expect(page.locator('h3.modal-title', has_text="新しいコンポーネントを作成する")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## タイトル「TEST-グループ管理連携機能検証-YYYYMMDD(本日の日付)-コンポーネント-ユーザー2」でコンポーネントを作成する
※上記のコンポーネントを作成する際、「次からメンバーとグループを追加する：{プロジェクト名}」にチェックを入れること

- 「新しいコンポーネントが作成されました！」ダイアログが表示されること

In [ ]:
async def _step(page):

    from datetime import datetime

    modal = page.locator("#addSubComponent")
    await expect(modal).to_be_visible(timeout=transition_timeout)

    # Input name
    title_input = modal.locator('input[name="projectName"]')
    await title_input.click()
    await title_input.type(component_title_display_username_2, delay=50)  
    await title_input.blur() 

    # Checkbox
    await modal.locator('input[name="inherit_contributors"]').check()

    # Đợi button 作成 enable
    create_btn = modal.locator('button.btn-success',has_text="作成")
    await expect(create_btn).to_be_enabled(timeout=transition_timeout)

    # Click
    await create_btn.click()

    # Success
    await expect(modal.locator('h4.add-project-success.text-success',has_text="新しいコンポーネントが作成されました！")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「新しいコンポーネントが作成されました！」ダイアログの「新しいコンポーネントへ移動する」をクリックする

- 作成したコンポーネントのプロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):

    # This link appears after creating a component
    move_link = page.locator('a.btn.btn-success', has_text="新しいコンポーネントへ移動する")

    # If the link exists, click it to navigate to the Project Dashboard
    if await move_link.count() > 0:
        await expect(move_link).to_be_visible(timeout=transition_timeout)
        await move_link.click()

    # Ensure modal is closed
        await expect(page.locator('.modal.fade.in')).to_have_count(0, timeout=transition_timeout)

    # Project dashboard is displayed
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)

    await page.locator('//h3[text()="最近の活動"]').click()

await run_pw(_step)

##  以下の手順を実施し、コンポーネントを削除する
-  コンポーネントのプロジェクトダッシュボードの上部メニューから「設定」をクリックする
- 「コンポーネントを削除」を選択する
- 「次の文字列を入力して続行します」に記載されている文字列を入力欄に記載し、「削除」をクリックする

- 以下を確認すること
    - 設定画面が表示されること
    - 「このコンポーネントを削除してもよろしいですか？」のダイアログが表示されること
    - プロジェクトダッシュボードの「コンポーネント」から該当コンポーネントが消えること

In [ ]:
async def _step(page):

    # Open 設定 screen
    await page.get_by_role("link", name="設定").click()
    
    # 設定 screen is displayed
    await expect(page.get_by_role("button", name="コンポーネントを削除")).to_be_visible(timeout=transition_timeout)

    # Click "コンポーネントを削除" button
    await page.get_by_role("button", name="コンポーネントを削除").click()

     # Wait for delete confirmation modal
    delete_modal = page.locator('#nodesDelete')
    await expect(delete_modal).to_be_visible(timeout=transition_timeout)

    await expect(delete_modal.locator('h3.modal-title')).to_have_text("このコンポーネントを削除してもよろしいですか？", timeout=transition_timeout)

    confirmation_label = page.locator('//strong[@data-bind = "text: confirmationString"]')
    await expect(confirmation_label).to_have_count(1, timeout=transition_timeout)
    confirmation = await confirmation_label.text_content()
    print(confirmation)

    await asyncio.sleep(1)
    confirmation_input = page.locator('//*[@data-bind = "editableHTML: {observable: confirmInput, onUpdate: handleEditableUpdate}"]')
    await confirmation_input.fill(confirmation)

    delete_button = page.locator('//a[contains(@class, "btn-danger") and text() = "削除"]')
    await expect(delete_button).to_be_visible()
    await delete_button.click()

    # Project dashboard is displayed
    await expect(page.locator('//a[text()="アドオン"]')).to_be_visible(timeout=transition_timeout)

    # Deleted component is not displayed
    await expect(page.locator('a', has_text=component_title_display_username_2)).to_have_count(0, timeout=transition_timeout)
    
await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「メンバー」をクリックする

- 以下を確認すること
    - 「メンバー」画面が表示されること
    - 各メンバの「権限」項目が変更できないこと
    - 既存ユーザー2以外のメンバーを削除できないこと

In [ ]:
async def _step(page):
    
    members_link = page.locator("#projectSubnav").get_by_role("link", name="メンバー")
    await expect(members_link).to_be_visible(timeout=transition_timeout)
    await members_link.click()

    # メンバー screen is displayed
    await expect(page.locator("h3", has_text="メンバー")).to_be_visible(timeout=transition_timeout)

    # Expect permission column is not editable
    await expect(page.locator('td.permissions select, td.permissions input')).to_have_count(0)

    # The delete icon is not displayed (except for User 2)
    await expect(page.locator('i.fa.fa-times.fa-2x.remove-or-reject')).to_be_visible();

await run_pw(_step)

## ユーザーメニューから「ログアウト」を選択する

- GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_1, transition_timeout=transition_timeout)

await run_pw(_step)

## ウェブブラウザの新規プライベートウィンドウで GRDM トップページを表示する

- GRDM トップページが表示されること

In [ ]:
import time

async def _step(page):
    await page.goto(rdm_url)

    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count():
        await consent_button.click()

    await expect(consent_button).to_have_count(0, timeout=500)

await run_pw(_step)

## RDMIdPを利用し、既存ユーザー3としてログインする

・GRDM ダッシュボードが表示されること   
・「プロジェクトに対するグループ機能①」シートのNo.4で追加したプロジェクトが表示されていること   
・該当プロジェクトの「Groups」項目に既存ユーザー3が所属しているグループ(グループB)が表示されていること   
※ メンバの表示件数は 3 件まで。4 名以上の場合、「他n名」が表示されること。  
※ グループの表示件数は 3 件まで。4 グループ以上の場合、「他nグループ」が表示されること。  

In [ ]:
async def _step(page):
    
    await grdm.login_after_logout(page, idp_name_1, idp_username_3, idp_password_3, transition_timeout=transition_timeout)

    # GRDMのボタンが表示されることを確認
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)
   
    project_locator = page.locator('[data-test-dashboard-item]', has=page.locator('[data-test-dashboard-item-title]', has_text=rdm_project_name))
    await expect(project_locator).to_be_visible(timeout=transition_timeout)
    await expect(project_locator).to_contain_text('他1グループ', timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードから「プロジェクトに対するグループ機能①」シートのNo.4で作成したプロジェクトをクリックする

- プロジェクトダッシュボードが表示されること
- プロジェクトダッシュボードの上部メニューに「ファイル」、「Wiki」、「メタデータ」、「エクスポート」、「メンバ」、「グループ」、「アドオン」、「設定」が表示されること(「読込み/書込み」権限)

In [ ]:
async def _step(page):

    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]').click()

    await expect(page.locator("#projectSubnav").get_by_role("link", name="ファイル")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="Wiki")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メタデータ")).to_be_visible(timeout=transition_timeout)
    # await expect(page.locator("#projectSubnav").get_by_role("link", name="エクスポート")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メンバー")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True)).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="アドオン")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="設定")).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)

await run_pw(_step)

## 画面上部の「マイプロジェクト」をクリックする

- マイプロジェクトが表示されること 
- 「プロジェクトに対するグループ機能①」シートのNo.4で追加したプロジェクトが表示されていること 
- 該当プロジェクトの「Groups」項目に既存ユーザー3が所属しているグループ(グループB)が表示されていること

In [ ]:
async def _step(page):
    
    await page.locator('//a[text()="マイプロジェクト"]').click()
    
    #マイプロジェクト一覧が表示されること
    await expect(page.locator("h3", has_text="マイプロジェクト")).to_be_visible(timeout=transition_timeout)

   #プロジェクトが表示されること
    await expect(page.locator(f'//a[@class="fg-file-links" and text()="{rdm_project_name}"]')).to_be_visible(timeout=transition_timeout)

   #グループ(グループA)が表示されていること
    import re

    project_row = page.locator(".tb-row").filter(has=page.locator(f'//a[@class="fg-file-links" and text()="{rdm_project_name}"]'))

    container = project_row.locator(".tb-td.tb-col-3")
    group_span = container.locator("span", has_text=group_b)
    plus_indicator = container.locator("span", has_text=re.compile(r'\+ \d+'))

    # Verify group name existence: either shown directly as a span, or folded into "+N" (overflow)
    # (plus_indicator being visible does not guarantee the target group is included in it,
    #  so check group_span first and only fall back to the +N possibility if not found)
    if await group_span.count() > 0 and await group_span.first.is_visible():
        # Group name span is present in DOM (may be shown in full or ellipsis-truncated)
        await expect(group_span.first).to_be_visible(timeout=transition_timeout)
    elif await plus_indicator.count() > 0 and await plus_indicator.first.is_visible():
        # Overflow groups are collapsed into "+N" (tooltip not checked)
        await expect(plus_indicator.first).to_be_visible(timeout=transition_timeout)
    else:
        # Neither span nor plus_indicator matched, but the text should still exist in the DOM, so verify it
        await expect(container).to_contain_text(group_b, timeout=transition_timeout)

    # Ellipsis (visual truncation) verification always runs as its own case, independent of the existence check above
    is_truncated = await container.evaluate(
        "el => el.scrollWidth > el.clientWidth"
    )

    # Independent signal: check the computed CSS truncation setup itself, not derived from is_truncated
    # (so this assertion can actually fail if the ellipsis CSS is broken/removed by a regression)
    has_ellipsis_style = await container.evaluate(
        "el => { const s = getComputedStyle(el); return s.textOverflow === 'ellipsis' && s.overflow === 'hidden'; }"
    )

    if is_truncated:
        # Text is long and overflows the column -> the ellipsis CSS must actually be applied to visually truncate it
        assert has_ellipsis_style, "Expected ellipsis CSS (text-overflow/overflow) to be applied when text overflows"
    else:
        # Text is short and fits the column -> no truncation needed, full text is shown as-is
        assert not is_truncated, "Expected short text NOT to be truncated (no ellipsis)"
await run_pw(_step)

## マイプロジェクトから「プロジェクトに対するグループ機能①」シートのNo.4で作成したプロジェクトをクリックする

- プロジェクトダッシュボードが表示されること
- プロジェクトダッシュボードの上部メニューに「ファイル」、「Wiki」、「メタデータ」、「エクスポート」、「メンバ」、「グループ」、「アドオン」、「設定」が表示されること(「読込み/書込み」権限)

In [ ]:
async def _step(page):
    
    await page.locator(f'//a[@class="fg-file-links" and text()="{rdm_project_name}"]').click()

    await expect(page.locator("#projectSubnav").get_by_role("link", name="ファイル")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="Wiki")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メタデータ")).to_be_visible(timeout=transition_timeout)
    # await expect(page.locator("#projectSubnav").get_by_role("link", name="エクスポート")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メンバー")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True)).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="アドオン")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="設定")).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)

await run_pw(_step)

## ユーザーメニューから「ログアウト」を選択する

- GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_1, transition_timeout=transition_timeout)

await run_pw(_step)

## ウェブブラウザの新規プライベートウィンドウで GRDM トップページを表示する

- GRDM トップページが表示されること

In [ ]:
import time

async def _step(page):
    await page.goto(rdm_url)

    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count():
        await consent_button.click()

    await expect(consent_button).to_have_count(0, timeout=500)

await run_pw(_step)

## RDMIdPを利用し、既存ユーザー4としてログインする

- GRDM ダッシュボードが表示されること 
- 「プロジェクトに対するグループ機能①」シートのNo.4で追加したプロジェクトが表示されていること 
- 該当プロジェクトの「Groups」項目に既存ユーザー4が所属しているグループ(グループC)が表示されていること

In [ ]:
async def _step(page):
    
    await grdm.login_after_logout(page, idp_name_1, idp_username_4, idp_password_4, transition_timeout=transition_timeout)

    # GRDMのボタンが表示されることを確認
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)
   
    project_locator = page.locator('[data-test-dashboard-item]', has=page.locator('[data-test-dashboard-item-title]', has_text=rdm_project_name))
    await expect(project_locator).to_be_visible(timeout=transition_timeout)
    await expect(project_locator).to_contain_text(group_c, timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードから「プロジェクトに対するグループ機能①」シートのNo.4で作成したプロジェクトをクリックする

- プロジェクトダッシュボードが表示されること
- プロジェクトダッシュボードの上部メニューに「ファイル」、「Wiki」、「メタデータ」、「エクスポート」、「メンバ」、「グループ」、「アドオン」、「設定」「証跡管理」が表示されること(「管理者」権限)

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]').click()

    await expect(page.locator("#projectSubnav").get_by_role("link", name="ファイル")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="Wiki")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メタデータ")).to_be_visible(timeout=transition_timeout)
    # await expect(page.locator("#projectSubnav").get_by_role("link", name="エクスポート")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メンバー")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True)).to_be_visible(timeout=transition_timeout)
    # await expect(page.locator("#projectSubnav").get_by_role("link", name="アドオン")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="設定")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="証跡管理")).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)
    
await run_pw(_step)

## 画面上部の「マイプロジェクト」をクリックする

- マイプロジェクトが表示されること
- 「プロジェクトに対するグループ機能①」シートのNo.4で追加したプロジェクトが表示されていること
- 該当プロジェクトの「Groups」項目に既存ユーザー4が所属しているグループが表示されていること

In [ ]:
async def _step(page):
    
    await page.locator('//a[text()="マイプロジェクト"]').click()
    
    #マイプロジェクト一覧が表示されること
    await expect(page.locator("h3", has_text="マイプロジェクト")).to_be_visible(timeout=transition_timeout)

   #プロジェクトが表示されること
    await expect(page.locator(f'//a[@class="fg-file-links" and text()="{rdm_project_name}"]')).to_be_visible(timeout=transition_timeout)

   #グループ(グループA)が表示されていること
    import re

    project_row = page.locator(".tb-row").filter(has=page.locator(f'//a[@class="fg-file-links" and text()="{rdm_project_name}"]'))

    container = project_row.locator(".tb-td.tb-col-3")
    group_span = container.locator("span", has_text=group_c)
    plus_indicator = container.locator("span", has_text=re.compile(r'\+ \d+'))

    # Verify group name existence: either shown directly as a span, or folded into "+N" (overflow)
    # (plus_indicator being visible does not guarantee the target group is included in it,
    #  so check group_span first and only fall back to the +N possibility if not found)
    if await group_span.count() > 0 and await group_span.first.is_visible():
        # Group name span is present in DOM (may be shown in full or ellipsis-truncated)
        await expect(group_span.first).to_be_visible(timeout=transition_timeout)
    elif await plus_indicator.count() > 0 and await plus_indicator.first.is_visible():
        # Overflow groups are collapsed into "+N" (tooltip not checked)
        await expect(plus_indicator.first).to_be_visible(timeout=transition_timeout)
    else:
        # Neither span nor plus_indicator matched, but the text should still exist in the DOM, so verify it
        await expect(container).to_contain_text(group_c, timeout=transition_timeout)

    # Ellipsis (visual truncation) verification always runs as its own case, independent of the existence check above
    is_truncated = await container.evaluate(
        "el => el.scrollWidth > el.clientWidth"
    )

    # Independent signal: check the computed CSS truncation setup itself, not derived from is_truncated
    # (so this assertion can actually fail if the ellipsis CSS is broken/removed by a regression)
    has_ellipsis_style = await container.evaluate(
        "el => { const s = getComputedStyle(el); return s.textOverflow === 'ellipsis' && s.overflow === 'hidden'; }"
    )

    if is_truncated:
        # Text is long and overflows the column -> the ellipsis CSS must actually be applied to visually truncate it
        assert has_ellipsis_style, "Expected ellipsis CSS (text-overflow/overflow) to be applied when text overflows"
    else:
        # Text is short and fits the column -> no truncation needed, full text is shown as-is
        assert not is_truncated, "Expected short text NOT to be truncated (no ellipsis)"
await run_pw(_step)

## マイプロジェクトから「プロジェクトに対するグループ機能①」シートのNo.4で作成したプロジェクトをクリックする

- プロジェクトダッシュボードが表示されること
- プロジェクトダッシュボードの上部メニューに「ファイル」、「Wiki」、「メタデータ」、「エクスポート」、「メンバ」、「グループ」、「アドオン」、「設定」「証跡管理」が表示されること(「管理者」権限)

In [ ]:
async def _step(page):
    
    await page.locator(f'//a[@class="fg-file-links" and text()="{rdm_project_name}"]').click()

    await expect(page.locator("#projectSubnav").get_by_role("link", name="ファイル")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="Wiki")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メタデータ")).to_be_visible(timeout=transition_timeout)
    # await expect(page.locator("#projectSubnav").get_by_role("link", name="エクスポート")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メンバー")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True)).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="アドオン")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="設定")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="証跡管理")).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)

await run_pw(_step)

## プロジェクトダッシュボードの画面右部の「コンポーネント」の「コンポーネントを追加」をクリックする

- 「新しいコンポーネントを作成する」ダイアログが表示されること

In [ ]:
async def _step(page):
    
    await page.locator('div[data-target="#addSubComponent"]').click()

    await expect(page.locator('h3.modal-title', has_text="新しいコンポーネントを作成する")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## タイトル「TEST-グループ管理連携機能検証-YYYYMMDD(本日の日付)-コンポーネント-ユーザー4」でコンポーネントを作成する
※上記のコンポーネントを作成する際、「次からメンバーとグループを追加する：{プロジェクト名}」にチェックを入れること

- 「新しいコンポーネントが作成されました！」ダイアログが表示されること

In [ ]:
async def _step(page):

    from datetime import datetime

    modal = page.locator("#addSubComponent")
    await expect(modal).to_be_visible(timeout=transition_timeout)

    # Input name
    title_input = modal.locator('input[name="projectName"]')
    await title_input.click()
    await title_input.type(component_title_display_username_4, delay=50)  
    await title_input.blur() 

    # Checkbox
    await modal.locator('input[name="inherit_contributors"]').check()

    # Đợi button 作成 enable
    create_btn = modal.locator('button.btn-success',has_text="作成")
    await expect(create_btn).to_be_enabled(timeout=transition_timeout)

    # Click
    await create_btn.click()

    # Success
    await expect(modal.locator('h4.add-project-success.text-success',has_text="新しいコンポーネントが作成されました！")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「新しいコンポーネントが作成されました！」ダイアログの「新しいコンポーネントへ移動する」をクリックする

- 作成したコンポーネントのプロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):

    # This link appears after creating a component
    move_link = page.locator('a.btn.btn-success', has_text="新しいコンポーネントへ移動する")

    # If the link exists, click it to navigate to the Project Dashboard
    if await move_link.count() > 0:
        await expect(move_link).to_be_visible(timeout=transition_timeout)
        await move_link.click()

    # Ensure modal is closed
        await expect(page.locator('.modal.fade.in')).to_have_count(0, timeout=transition_timeout)

    # Project dashboard is displayed
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)

    await page.locator('//h3[text()="最近の活動"]').click()

await run_pw(_step)

## 以下の手順を実施し、コンポーネントを削除する
- プロジェクトダッシュボードの上部メニューから「設定」をクリックする
- 「コンポーネントを削除」を選択する
- 「次の文字列を入力して続行します」に記載されている文字列を入力欄に記載し、「削除」をクリックする

- 以下を確認すること
    - 設定画面が表示されること
    - 「このコンポーネントを削除してもよろしいですか？」のダイアログが表示されること
    - プロジェクトダッシュボードの「コンポーネント」から該当コンポーネントが消えること

In [ ]:
async def _step(page):

    # Open 設定 screen
    await page.get_by_role("link", name="設定").click()
    
    # 設定 screen is displayed
    await expect(page.get_by_role("button", name="コンポーネントを削除")).to_be_visible(timeout=transition_timeout)

    # Click "コンポーネントを削除" button
    await page.get_by_role("button", name="コンポーネントを削除").click()

     # Wait for delete confirmation modal
    delete_modal = page.locator('#nodesDelete')
    await expect(delete_modal).to_be_visible(timeout=transition_timeout)

    await expect(delete_modal.locator('h3.modal-title')).to_have_text("このコンポーネントを削除してもよろしいですか？", timeout=transition_timeout)

    confirmation_label = page.locator('//strong[@data-bind = "text: confirmationString"]')
    await expect(confirmation_label).to_have_count(1, timeout=transition_timeout)
    confirmation = await confirmation_label.text_content()
    print(confirmation)

    # Input confirmation text
    await asyncio.sleep(1)
    confirmation_input = page.locator('//*[@data-bind = "editableHTML: {observable: confirmInput, onUpdate: handleEditableUpdate}"]')
    await confirmation_input.fill(confirmation)

    # Click delete button
    delete_button = page.locator('//a[contains(@class, "btn-danger") and text() = "削除"]')
    await expect(delete_button).to_be_visible()
    await delete_button.click()

    # Project dashboard is displayed
    await expect(page.locator('//a[text()="アドオン"]')).to_be_visible(timeout=transition_timeout)

    # Deleted component is not displayed
    await expect(page.locator('a', has_text=component_title_display_username_4)).to_have_count(0, timeout=transition_timeout)
    
await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「メンバー」をクリックする

- 以下を確認すること
    - 「メンバー」画面が表示されること
    - 各メンバの「権限」項目が変更できること ( 「権限」項目がリスト形式で表示されていること)
    - メンバーを削除できること ( 「名前」項目の列の右端の「×」ボタンが表示されていること)

In [ ]:
async def _step(page):
    
    # Access メンバー screen
    members_link = page.locator("#projectSubnav").get_by_role("link", name="メンバー")
    await expect(members_link).to_be_visible(timeout=transition_timeout)
    await members_link.click()

    # メンバー screen is displayed
    await expect(page.locator("h3", has_text="メンバー")).to_be_visible(timeout=transition_timeout)

    # Count contributors
    contributors = page.locator('#manageContributorsTable tbody tr.contrib')
    count = await contributors.count()
    
    # Each row: permission dropdown is enabled
    for i in range(count):
        row = contributors.nth(i)
        permission_select = row.locator('td.permissions select')
        await expect(permission_select).to_be_visible(timeout=transition_timeout)
        await expect(permission_select).to_be_enabled(timeout=transition_timeout)

    # Each row: remove (X) button is visible
    remove_button = row.locator('td.add-remove i.remove-or-reject')
    await expect(remove_button).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「グループ」をクリックする

- 「グループ」画面が表示されること
- 各グループが以下の通りとなっていること。
    - 既存ユーザー2 が所属するグループ(グループA): 読込み/書込み
    - 既存ユーザー3 が所属するグループ(グループB): 読込み
    - 既存ユーザー4 が所属するグループ(グループC): 管理者
    - 既存ユーザー5 が所属するグループ(グループD): 読込み

In [ ]:
async def _step(page):
    
    # Access グループ screen
    await page.get_by_role("link", name="グループ", exact=True).click()

    # グループ screen is displayed
    await expect(page.get_by_role("heading", name=group_note_text)).to_be_visible(timeout=transition_timeout)

    # Groups permission
    Groups = {group_a: "読込み / 書込み", group_b: "読込み", group_c: "管理者", group_d: "読込み",}

    groups_tbody = page.locator("#groups")

    for group_name, expected_permission in Groups.items():
        row = groups_tbody.locator("tr").filter(has=page.get_by_text(group_name, exact=True))
        await expect(row).to_have_count(1)
        permission = row.locator(".permission-filter")
        await expect(permission).to_have_text(expected_permission)

await run_pw(_step)

## 「名前」項目の既存ユーザー4が所属するグループ(グループC)の列の右端の「×」ボタンをクリックする

- 「グループを削除」ダイアログが表示されること

In [ ]:
async def _step(page):
    
    # Identify group C
    group_row = page.locator("#groups tr").filter(has=page.get_by_text(group_c, exact=True))
    
    # Click the "remove (×)" button
    remove_button = group_row.locator("td.add-remove i.remove-or-reject")
    await asyncio.sleep(1)
    await remove_button.click() 
    await expect(page.locator("h3.modal-title", has_text="グループを削除")).to_be_visible(timeout=transition_timeout*2)

await run_pw(_step)

## 「グループを削除」ダイアログの「削除」ボタンをクリックする

- 「グループ」画面が表示されること
- 各グループが以下の通りとなっていること。
    - 既存ユーザー2 が所属するグループ(グループA): 読込み/書込み
    - 既存ユーザー3 が所属するグループ(グループB): 読込み
    - 既存ユーザー5 が所属するグループ(グループD): 読込み"

In [ ]:
async def _step(page):
    # Modal is already open from previous testcase
    delete_modal = page.locator("div.modal-content.scripted", has=page.locator("h3.modal-title", has_text="グループを削除"))

    # Ensure modal exists (not visibility animation dependent)
    await expect(delete_modal).to_be_attached()

    # Click 削除 button
    delete_button = delete_modal.locator("div.remove-page-buttons a.btn.btn-danger")
    await delete_button.click()

    error_heading = page.locator("h2#error[data-http-status-code='403']", has_text="Forbidden")
    await expect(error_heading).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ユーザーメニューから「ログアウト」を選択する

- GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_1, transition_timeout=transition_timeout)

await run_pw(_step)

## RDMIdPを利用し、機関管理者1としてログインする
(試験条件を分かりやすくするために、機関管理者アカウントを利用して、ユーザー画面にログインする)

- GRDM ダッシュボードが表示されること

In [ ]:
async def _step(page):
    
    await grdm.login_after_logout(page, idp_name_1, idp_username_institutional_admin, idp_password_institutional_admin, transition_timeout=transition_timeout)

    # GRDMのボタンが表示されることを確認
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「グループ」をクリックする

- 「グループ」画面が表示されること
- 各グループが以下の通りとなっていること。
    - 既存ユーザー2 が所属するグループ(グループA): 読込み/書込み
    - 既存ユーザー3 が所属するグループ(グループB): 読込み
    - 既存ユーザー5 が所属するグループ(グループD): 読込み

In [ ]:
async def _step(page):

    # Project Dashboard should be displayed
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]').click()        
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)

    # Access グループ screen
    await page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True).click()

    # グループ screen is displayed
    await expect(page.get_by_role("heading", name=group_note_text)).to_be_visible(timeout=transition_timeout)

    groups = [(group_a, "読込み / 書込み"),(group_b, "読込み"),(group_d, "読込み"),]
    for group_name, permission in groups:
        group_row = page.locator("#manageGroupsTable tbody tr.contrib", has=page.locator("a.name-search", has_text=group_name))

    # グループが表示されていること
    await expect(group_row).to_be_visible()

    # 権限が正しいこと
    await expect(group_row.locator("span.permission-search")).to_have_text(permission)
   
await run_pw(_step)

## 「グループ」のタイトルの横にある「＋追加」をクリックする

- 「グループを追加」ダイアログが表示されること

In [ ]:
async def _step(page):
    
    # Click 追加 button
    add_button = page.locator('a[href="#addGroups"]', has_text="追加")
    await expect(add_button).to_be_visible(timeout=transition_timeout)
    await add_button.click()

    # Add Group dialog is displayed
    add_group_modal_title = page.locator('div#addGroups h3.modal-title',has_text="グループを追加")
    await expect(add_group_modal_title).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 既存ユーザー4が所属するグループ(グループC)を入力して、「検索」ボタンをクリックする

- 「結果」の一覧に既存ユーザー4が所属するグループ(グループC)が表示されること
- 「結果」の一覧に既存ユーザー2が所属するグループ(グループA)、既存ユーザー3が所属するグループ(グループB)、既存ユーザー5が所属するグループ(グループD)が「✔️(追加済)」として表示されること

In [ ]:
async def _step(page):

    # Search input
    search_input = page.locator('#addGroups input[placeholder="グループ名で検索する"]')
    await expect(search_input).to_be_visible(timeout=transition_timeout)
    await search_input.fill("group")

    # Click 検索
    search_button = page.locator('#addGroups input[type="submit"][value="検索"]')
    await expect(search_button).to_be_visible(timeout=transition_timeout)
    await search_button.click()

    # Wait results appear 
    results_rows = page.locator('#addGroups tbody tr')
    await expect(results_rows.first).to_be_visible(timeout=transition_timeout)

    # Verify search results: Group C is addable, Groups A, B, and D are already added
    async def expect_group_added(group_name):
        row = page.locator('#addGroups tbody tr').filter(has=page.get_by_text(group_name, exact=True))
        await expect(row).to_be_visible()
        await expect(row.locator('i.fa-check')).to_be_visible()
        await expect(row.locator('i.fa-plus')).not_to_be_visible()

    async def expect_group_addable(group_name):
        row = page.locator('#addGroups tbody tr').filter(has=page.get_by_text(group_name, exact=True))
        await expect(row).to_be_visible()
        await expect(row.locator('i.fa-plus')).to_be_visible()
        await expect(row.locator('i.fa-check')).not_to_be_visible()

    await expect_group_addable(group_c)
    await expect_group_added(group_a)
    await expect_group_added(group_b)
    await expect_group_added(group_d)

await run_pw(_step)

## 「結果」の一覧の既存ユーザー4が所属するグループ(グループC)の横の「＋」ボタンをクリックする

- 「追加中」の一覧に既存ユーザー4が所属するグループ(グループC)が表示されること

In [ ]:
async def _step(page):

    # Identify group C
    group_row = page.locator('#addGroups tbody tr').filter(has=page.get_by_text(group_c, exact=True))
    await expect(group_row).to_be_visible(timeout=transition_timeout)

    # Click + button
    add_button = group_row.locator('a:has(i.fa-plus)')
    await expect(add_button).to_be_visible(timeout=transition_timeout)
    await add_button.click()

    # Group C appears in 追加中
    adding_section = page.locator('div.col-md-8', has=page.get_by_text("追加中", exact=True))
    group_c_row = adding_section.locator('tbody tr', has=page.get_by_text(group_c, exact=True))
    await expect(group_c_row).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「権限」を「管理者」に設定して、「追加」ボタンをクリックする

- 「グループ」画面に既存ユーザー4が所属するグループ(グループC)が追加されること
- 「グループ」画面に既存ユーザー4が所属するグループ(グループC)の「権限」が「管理者」に設定されていること

In [ ]:
async def _step(page):
    
    # Group C row in 追加中
    group_c_row = page.locator('div.col-md-8 tbody tr', has=page.get_by_text(group_c, exact=True))
    await expect(group_c_row).to_be_visible(timeout=transition_timeout)

    # Select 管理者
    permission_select = group_c_row.locator('select')
    await permission_select.select_option(label="管理者")

    # Click 追加 button in popup
    add_button = page.locator('div.modal-footer a.btn.btn-success', has_text="追加")
    await expect(add_button).to_be_visible(timeout=transition_timeout)
    await expect(add_button).to_be_enabled()
    await add_button.click()

    #Identify Group C row in グループ table
    group_c_row = page.locator('#groups tr').filter(has=page.get_by_text(group_c, exact=True))

    # Group C is displayed
    await expect(group_c_row).to_be_visible(timeout=transition_timeout)

    # Permission is 管理者
    permission = group_c_row.locator("span.permission-search")
    await expect(permission).to_have_text("管理者")

await run_pw(_step)

## ユーザーメニューから「ログアウト」を選択する

- GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_1, transition_timeout=transition_timeout)

await run_pw(_step)

## ウェブブラウザの新規プライベートウィンドウで GRDM トップページを表示する

- GRDM トップページが表示されること

In [ ]:
import time

async def _step(page):
    await page.goto(rdm_url)

    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count():
        await consent_button.click()

    await expect(consent_button).to_have_count(0, timeout=500)

await run_pw(_step)

## RDMIdPを利用し、既存ユーザー5としてログインする

- GRDM ダッシュボードが表示されること
- 「プロジェクトに対するグループ機能①」シートのNo.4で追加したプロジェクトが表示されていること
- 該当プロジェクトの「Groups」項目に既存ユーザー5が所属しているグループ(グループD)が表示されていること

In [ ]:
async def _step(page):
    await grdm.login_after_logout(page, idp_name_1, idp_username_5, idp_password_5, transition_timeout=transition_timeout)

    # GRDMのボタンが表示されることを確認
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)
   
    project_locator = page.locator('[data-test-dashboard-item]', has=page.locator('[data-test-dashboard-item-title]', has_text=rdm_project_name))
    await expect(project_locator).to_be_visible(timeout=transition_timeout)
    await expect(project_locator).to_contain_text(group_d, timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードから「プロジェクトに対するグループ機能①」シートのNo.4で作成したプロジェクトをクリックする

- プロジェクトダッシュボードが表示されること
- プロジェクトダッシュボードの上部メニューに「ファイル」、「Wiki」、「メタデータ」、「エクスポート」、「メンバー」、「グループ」、「設定」が表示されること(「読込み」権限)

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]').click()        

    await expect(page.locator("#projectSubnav").get_by_role("link", name="ファイル")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="Wiki")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メタデータ")).to_be_visible(timeout=transition_timeout)
    # await expect(page.locator("#projectSubnav").get_by_role("link", name="エクスポート")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メンバー")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True)).to_be_visible(timeout=transition_timeout)
    # await expect(page.locator("#projectSubnav").get_by_role("link", name="アドオン")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="設定")).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)

await run_pw(_step)

## 画面上部の「マイプロジェクト」をクリックする

- マイプロジェクトが表示されること
- 「プロジェクトに対するグループ機能①」シートのNo.4で追加したプロジェクトが表示されていること
- 該当プロジェクトの「Groups」項目に既存ユーザー5が所属しているグループ(グループD)が表示されていること

In [ ]:
async def _step(page):
    
    await page.locator('//a[text()="マイプロジェクト"]').click()
    
    #マイプロジェクト一覧が表示されること
    await expect(page.locator("h3", has_text="マイプロジェクト")).to_be_visible(timeout=transition_timeout)

   #プロジェクトが表示されること
    await expect(page.locator(f'//a[@class="fg-file-links" and text()="{rdm_project_name}"]')).to_be_visible(timeout=transition_timeout)

   #グループ(グループA)が表示されていること
    import re

    project_row = page.locator(".tb-row").filter(has=page.locator(f'//a[@class="fg-file-links" and text()="{rdm_project_name}"]'))

    container = project_row.locator(".tb-td.tb-col-3")
    group_span = container.locator("span", has_text=group_d)
    plus_indicator = container.locator("span", has_text=re.compile(r'\+ \d+'))

    # Verify group name existence: either shown directly as a span, or folded into "+N" (overflow)
    # (plus_indicator being visible does not guarantee the target group is included in it,
    #  so check group_span first and only fall back to the +N possibility if not found)
    if await group_span.count() > 0 and await group_span.first.is_visible():
        # Group name span is present in DOM (may be shown in full or ellipsis-truncated)
        await expect(group_span.first).to_be_visible(timeout=transition_timeout)
    elif await plus_indicator.count() > 0 and await plus_indicator.first.is_visible():
        # Overflow groups are collapsed into "+N" (tooltip not checked)
        await expect(plus_indicator.first).to_be_visible(timeout=transition_timeout)
    else:
        # Neither span nor plus_indicator matched, but the text should still exist in the DOM, so verify it
        await expect(container).to_contain_text(group_d, timeout=transition_timeout)

    # Ellipsis (visual truncation) verification always runs as its own case, independent of the existence check above
    is_truncated = await container.evaluate(
        "el => el.scrollWidth > el.clientWidth"
    )

    # Independent signal: check the computed CSS truncation setup itself, not derived from is_truncated
    # (so this assertion can actually fail if the ellipsis CSS is broken/removed by a regression)
    has_ellipsis_style = await container.evaluate(
        "el => { const s = getComputedStyle(el); return s.textOverflow === 'ellipsis' && s.overflow === 'hidden'; }"
    )

    if is_truncated:
        # Text is long and overflows the column -> the ellipsis CSS must actually be applied to visually truncate it
        assert has_ellipsis_style, "Expected ellipsis CSS (text-overflow/overflow) to be applied when text overflows"
    else:
        # Text is short and fits the column -> no truncation needed, full text is shown as-is
        assert not is_truncated, "Expected short text NOT to be truncated (no ellipsis)"
await run_pw(_step)

## マイプロジェクトから「プロジェクトに対するグループ機能①」シートのNo.4で作成したプロジェクトをクリックする

- プロジェクトダッシュボードが表示されること
- プロジェクトダッシュボードの上部メニューに「ファイル」、「Wiki」、「メタデータ」、「エクスポート」、「メンバー」、「グループ」、「設定」が表示されること(「読込み」権限)

In [ ]:
async def _step(page):
    
    await page.locator(f'//a[@class="fg-file-links" and text()="{rdm_project_name}"]').click()

    await expect(page.locator("#projectSubnav").get_by_role("link", name="ファイル")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="Wiki")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メタデータ")).to_be_visible(timeout=transition_timeout)
    # await expect(page.locator("#projectSubnav").get_by_role("link", name="エクスポート")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="メンバー")).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="グループ", exact=True)).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#projectSubnav").get_by_role("link", name="設定")).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)

await run_pw(_step)

## ユーザーメニューから「ログアウト」を選択する

- GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_1, transition_timeout=transition_timeout)

await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}